# MORPC Insights - Alternative Fuel Stations

## Overview

The [Alternative Fuels Data Center](https://afdc.energy.gov/) (a division of the U.S. Dept. of Energy) maintains a database containing the locations and attributes of alternative fuel stations. This notebook produces a tileset that includes a summary of alternative fuel stations for the MORPC 15-county region and the counties and communities therein.  This notebook is the final stage in a pipeline that fetches, standardizes, and summarizes the alternative fuel station data.

## Setup

### Load required libraries

In [ ]:
import pandas as pd
import os
import json
import datetime
import textwrap
import matplotlib
from matplotlib import pyplot as plt
import morpc
import logging

### Start logging

In [ ]:
morpc.logs.config_logs()
logger = logging.getLogger(__name__)

### User-specified parameters

In [ ]:
# Years of data to be included in output as a list of two integers (first year, last year)
YEAR_RANGE = [2000, 2025]

### Static parameters

In [ ]:
# This script may pull data from outputs of upstream workflows.  The locations of these outputs are specified by their path relative
# a GitHub root directory. This is a single directory which is presumed to contain local working copies of MORPC GitHub repositories.
# Specify the path to the directory on your system where the local working copies are stored. By default, the GitHub root directory is
# assumed to be one level up from this script.
GITHUB_ROOT = "../"

# Specify the path to the directory where the input data is stored. Sometimes the data may be sourced from this location and sometimes 
# it may be sourced from elsewhere and archived here.
INPUT_DIR = "./input_data"

# Specify the path to the directory where the output data is stored. Typically it is not necessary to change this, and changing it for 
# established scripts may break other scripts that depend on outputs from this one.
OUTPUT_DIR = "./output_data"

# Subdirectory of OUTPUT_DIR where charts will be created
CHART_DIRNAME = "charts"

# Set EXPORT_NOTEBOOK_AS_HTML to True to automatically export the notebook in HTML format when the script finishes.  Set to False to skip the export
EXPORT_NOTEBOOK_AS_HTML = True

### Define inputs

#### Create input data directory

Create input data directory if it doesn't exist.

In [ ]:
inputDir = os.path.normpath(INPUT_DIR)
if not os.path.exists(inputDir):
    os.makedirs(inputDir)

#### Standardized alternative fuel station data

In [ ]:
STATIONS_INPUT_TABLE_RESOURCE = os.path.normpath(os.path.join(GITHUB_ROOT, "morpc-altfuelstations-summarize/output_data/morpc-altfuelstations-all-long.resource.yaml"))
logger.info("Resource file: {}".format(STATIONS_INPUT_TABLE_RESOURCE))

#### Geography lookup table [375]

In [ ]:
GEOS_LOOKUP_TABLE_RESOURCE = os.path.normpath(os.path.join(GITHUB_ROOT, "morpc-geos-collect/output_data/morpc-geos-lookup.resource.yaml"))
logger.info("Resource file: {}".format(GEOS_LOOKUP_TABLE_RESOURCE))

### Define outputs

#### Create output data directory

Create output data directory if it doesn't exist.

In [ ]:
outputDir = os.path.normpath(OUTPUT_DIR)
if not os.path.exists(outputDir):
    os.makedirs(outputDir)   

Create chart directory if it doesn't exist.

In [ ]:
chartDir = os.path.join(outputDir, CHART_DIRNAME)
if not os.path.exists(chartDir):
    os.makedirs(chartDir)    

#### Alternative fuel stations by year

In [ ]:
ALTFUELSTATIONS_TABLE_FILENAME = "altfuelstations_long.csv"
ALTFUELSTATIONS_TABLE_PATH = os.path.join(outputDir, ALTFUELSTATIONS_TABLE_FILENAME)
ALTFUELSTATIONS_TABLE_SCHEMA_PATH = ALTFUELSTATIONS_TABLE_PATH.replace(".csv",".schema.yaml")
ALTFUELSTATIONS_TABLE_RESOURCE_PATH = ALTFUELSTATIONS_TABLE_PATH.replace(".csv",".resource.yaml")
logger.info("Data: {}".format(ALTFUELSTATIONS_TABLE_PATH))
logger.info("Schema: {}".format(ALTFUELSTATIONS_TABLE_SCHEMA_PATH))
logger.info("Resource file: {}".format(ALTFUELSTATIONS_TABLE_RESOURCE_PATH))

## Prepare input data

### Load geography lookup table

Load and validate data.

In [ ]:
(geos, geosResource, geosSchema) = morpc.frictionless.load_data(GEOS_LOOKUP_TABLE_RESOURCE, archiveDir=inputDir)

Inspect the data.

In [ ]:
geos.head()

### Load standardized fuel station data from upstream workflows

Load and validate data.

In [ ]:
(stationsRaw, stationsRawResource, stationsRawSchema) = morpc.frictionless.load_data(STATIONS_INPUT_TABLE_RESOURCE, archiveDir=inputDir)

Inspect the data.

In [ ]:
stationsRaw.head()

## Transform fuel station data to format required by Insights platform

Load the output schema.

In [ ]:
stationsSchema = morpc.frictionless.load_schema(ALTFUELSTATIONS_TABLE_SCHEMA_PATH)

Create a working copy.

In [ ]:
stations = stationsRaw.copy()

Extract data for the specified years.

In [ ]:
stations = stations.loc[stations["open_year"].isin(range(YEAR_RANGE[0], YEAR_RANGE[1]+1))].copy()

Extract the geographic summary level from the GEOID as a separate column.  Create another column containing human-readable descriptions of the geography levels.

In [ ]:
stations["SUMLEVEL"] = stations["GEOIDFQ"].apply(lambda x:x[0:3])
stations["GEOTYPE"] = stations["SUMLEVEL"].map(morpc.HIERARCHY_STRING_LOOKUP)

Join geography metadata from the geography lookup table, merging on GEOID.  Replace the geography name included in the fuel station data with the name from the lookup table.

In [ ]:
stations = stations.drop(columns="NAME").merge(geos[["GEOIDFQ","COUNTYFP","NAME","MUNITYPE"]], on="GEOIDFQ", how="left")

Add the name of the county where each station is located by way of the unique GEOID for the county.

In [ ]:
stations["COUNTYID"] = "39" + stations["COUNTYFP"]
stations["COUNTY"] = stations["COUNTYID"].map(morpc.CONST_COUNTY_ID_TO_NAME)

Update the names for townships only to include a suffix which identifies the geography as a township and appends the name of the county where the township is located.  This is necessary to disambiguate cases where the same township name is used in multiple counties.

In [ ]:
temp = stations.loc[stations["MUNITYPE"] == "Township"].copy()
temp["NAME"] = temp["NAME"] + " Township (" + temp["COUNTY"] + ")"
stations.update(temp["NAME"], overwrite=True, errors="ignore")

Update the names for counties to identify them as such.  This is necessary to disambiguate cases where a place and a county have the same name (e.g. Delaware).

In [ ]:
temp = stations.loc[stations["SUMLEVEL"] == "050"].copy()
temp["NAME"] = temp["NAME"] + " County"
stations.update(temp["NAME"], overwrite=True, errors="ignore")

Rename certain columns to make their names more human-readable.

In [ ]:
stations = stations.rename(columns={
    "open_year":"Open year",
    "fuel_type_code":"Fuel type"
})
                           

Convert the codes used for different fuel types to descriptions that are more human-readable.

In [ ]:
stations["Fuel type"] = stations["Fuel type"].map({
    "BD":"Biodiesel",
    "CNG":"Compressed Natural Gas (CNG)",
    "ELEC":"Electric",
    "E85":"Ethanol (E85)",
    "HY":"Hydrogen",
    "LNG":"Liquefied Natural Gas (LNG)",
    "LPG":"Propane (LPG)"
})

Extract only the columns required by the schema.

In [ ]:
stations = stations.filter(items=stationsSchema.field_names, axis="columns")

Cast the fields as the data types required by the schema.

In [ ]:
stations = morpc.frictionless.cast_field_types(stations, stationsSchema)

Sort the data by the fields by GEOID, then year opened, then fuel type, as specified by the primary key defined in the schema.

In [ ]:
stations = stations.sort_values(stationsSchema.primary_key)

Inspect the data.

In [ ]:
stations

## Export data

Write data to disk.

In [ ]:
stations.to_csv(ALTFUELSTATIONS_TABLE_PATH, index=False)

Create resource file and validate data.

In [ ]:
stationsResource = morpc.frictionless.create_resource(ALTFUELSTATIONS_TABLE_FILENAME, 
    resourcePath=ALTFUELSTATIONS_TABLE_RESOURCE_PATH,
    title="MORPC Insights | Alternative Fuel Stations by Year", 
    name="altfuelstations", 
    description="Count of Central Ohio refueling stations selling alternative fuels which opened in each year according to data maintained by the U.S. Dept of Energy Alternative Fuels Data Center.",
    writeResource=True,
    validate=True
)

## Generate static charts

In [ ]:
%matplotlib agg

for f in os.scandir(chartDir):
    os.remove(f)

# Load a standard color set for the chart elements.  Remove the first element to change up
# the sequence a bit to distinguish these tiles from others
colorset = json.loads(json.dumps(morpc.CONST_COLOR_CYCLES["matplotlib"]))
colorset.pop(0)

# Create a list to accumulate geographies for which a thumbnail is generated
platformIncludeList = []
# Iterate over each geography in data set
for geoid in stations["GEOIDFQ"].unique():
    
    # Extract the data for a single geography
    temp = stations.loc[stations["GEOIDFQ"] == geoid].copy()

    if(temp.empty):
        continue
        
    platformIncludeList.append(geoid)

    # Generate a title string
    geoName = temp.iloc[0]["NAME"]
    title = "Alternative Fuel Stations by Year Opened - {}".format(geoName)

    # Drop the geography name and type
    temp = temp.filter(items=["Open year","Fuel type","count"], axis="columns")
    
    # Pivot to wide format
    temp = temp.pivot(index="Open year", columns="Fuel type").reset_index()
    
    # Create and annotate the plot
    PLOTWIDTH = 8
    fig,ax = plt.subplots(figsize=(PLOTWIDTH,PLOTWIDTH/16*9))

    temp.plot.bar(ax=ax, x="Open year", y="count", stacked=True, color=colorset)
    temp = stations.loc[stations["GEOIDFQ"] == geoid].copy()
    
    ax.set_title(title, fontsize=14)
    xlabel = None
    ylabel = None
    ax.set_xlabel(None)
    ax.set_ylabel(None)
    ax.set_yticks([round(tick,0) for tick in ax.get_yticks()])
    handles, labels = ax.get_legend_handles_labels()
    labels = [textwrap.fill(label, 15) for label in labels]
    legend = ax.legend(labels, loc='center left', bbox_to_anchor=(1, 0.5), labelspacing=1)
    ax.grid(visible=True, color="lightgrey")
    ax.set_axisbelow(True)
    
    # Format the y-axis labels as integers with comma separators
    ax.get_yaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    
    # Save the figure to disk as an SVG file
    ax.figure.savefig(os.path.join(chartDir, "{}.svg".format(geoid)), bbox_extra_artists=(legend,), bbox_inches='tight')
    ax.figure.savefig(os.path.join(chartDir, "{}.png".format(geoid)), bbox_extra_artists=(legend,), bbox_inches='tight')
    
    plt.close(ax.figure)

    excelData = temp[["Open year","Fuel type","count"]].pivot(index="Open year", columns="Fuel type", values="count")

    writer = pd.ExcelWriter(os.path.join(chartDir, "{}.xlsx".format(geoid)), engine='xlsxwriter')
    dataOptions = {
        "numberFormat": {
            'Open year': "0",
            'Biodiesel': "#,##0",
            'Compressed Natural Gas (CNG)': "#,##0",
            'Electric': "#,##0",
            'Ethanol (E85)': "#,##0",
            'Hydrogen': "#,##0",
            'Liquefied Natural Gas (LNG)': "#,##0",
            'Propane (LPG)': "#,##0"
        },
        "columnWidth": 30
    }
    chartOptions = {
        "subtype":"stacked",
        "colors": colorset,
        "titles": {
            "chartTitle": title,
            "xTitle": xlabel,
            "yTitle": ylabel
        },
        "seriesOptions": [{"gap":100} for x in excelData.columns],
        "xAxisOptions": {
            "num_font": {"size":14},
        },
        "yAxisOptions": {
            "num_font": {"size":14},
            "num_format": "0",
        },
        "legendOptions":{
            "position":"bottom",
            "font":{"size":14}
        },
        "sizeOptions":{
            "x_scale":1.5,
            "y_scale":1.5
        }
    }
    morpc.data_chart_to_excel(excelData, writer, chartType="column", dataOptions=dataOptions, chartOptions=chartOptions)
    writer.close()

%matplotlib inline

## Generate Insights catalog content

The content in the Insights platform is controlled by a catalog spreadsheet. Each tile to be displayed in the platform must have a record in the catalog.  This section will create the records for the tiles that display the alternative fuel station data.  Eventually this function will be performed by a separate staging script.

First specify the column names used in the catalog.

In [ ]:
columnNames=["TileID","TilesetID","GeographyType","GeographyName","Category","Headline","Commentary","ThumbnailURL","Contributor","Vintage","UpdateInterval","ShareURL","DataProductURL","MoreInformationURL"]

Create a new dataframe containing only the geographies for which thumbnail images were produced in the section above.

In [ ]:
catalog = stations.loc[stations["GEOIDFQ"].isin(platformIncludeList)].copy()

Extract only the metadata columns of interest and flatten the data to have only one record per geography. Rename the metadata fields to match the catalog fields.

In [ ]:
catalog = catalog.filter(items=["GEOIDFQ","NAME","GEOTYPE"], axis="columns") \
    .groupby("GEOIDFQ").first() \
    .reset_index() \
    .rename(columns={"NAME":"GeographyName","GEOTYPE":"GeographyType"})

Change the GeographyType values to match the schema of the catalog.

In [ ]:
catalog["GeographyType"] = catalog["GeographyType"].map({
    "REGION15":"Region",
    "COUNTY":"County",
    "JURIS":"Community"
})

Populate some placeholder fields.

In [ ]:
catalog["TileID"] = None
catalog["TilesetID"] = None
catalog["Category"] = None
catalog["Headline"] = "TBD"
catalog["Commentary"] = "TBD"

Generate the URL for the thumbnail images. These will be hosted in GitHub and will be indexed by GEOIDFQ.

In [ ]:
catalog["ThumbnailURL"] = catalog["GEOIDFQ"].apply(lambda geoid:"https://raw.githubusercontent.com/morpc-insights/altfuelstations/refs/heads/main/output_data/charts/{}.svg".format(geoid))

Populate some other simple metadata.  Vintage in this case refers to the year that the content was published in Insights. This is to give readers an idea of how old it is.  UpdateInterval gives them an idea of when to expect the next version. ShareURL is a placeholder for now.

In [ ]:
catalog["Contributor"] = "Mid-Ohio Regional Planning Commission"
catalog["Vintage"] = str(datetime.date.today().year)
catalog["UpdateInterval"] = "annually"
catalog["ShareURL"] = None
catalog["MoreInformationURL"] = None

Generate the data product URL.  This points to an ArcGIS Dashboard that accepts URL parameters.  GEOIDFQ is passed as a parameter to tell the app to load the data for a particular geography.

In [ ]:
catalog["DataProductURL"] = catalog["GEOIDFQ"].apply(lambda geoid:"https://www.arcgis.com/apps/dashboards/87fd6a0b569b430c9e4c6e7fa4fbd4c5#geoid={}".format(geoid))

Extract only the required columns.

In [ ]:
catalog = catalog.filter(items=columnNames, axis="columns")

Inspect the listing.

In [ ]:
catalog.head()

Save the catalog to an Excel spreadsheet.

In [ ]:
catalog.to_excel("catalog.xlsx", index=False)

It is necessary to manually add these records to the master catalog or update the records already therein.  See the following file in GitHub:

https://github.com/morpc/morpc-insights/blob/main/catalog/morpc_insights_catalog.xlsx


## Export notebook as HTML

In [ ]:
if(EXPORT_NOTEBOOK_AS_HTML):
    logger.info("Exporting notebook to HTML format")    
    morpc.notebook_to_html()
else:
    logger.info("Skipping export to HTML format")    